<a href="https://colab.research.google.com/github/Angela-Benedict/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path = 'lines', meta=['order', 'vendor_id'
  ], errors='ignore')
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
assert len(lines)==4
assert sum(lines['qty'])
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors_json_new = pd.DataFrame(vendors_json)
merged = pd.merge(lines, vendors_json_new, on = 'vendor_id', how = 'left', indicator=True)
print(merged)
print()
print('Vendor V-07 does not match, and 3 units are attatched to it.')

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

Vendor V-07 does not match, and 3 units are attatched to it.


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
merged['name'] = merged['name'].fillna('Unknown')
name_total_quantity = merged.groupby('name', as_index=False)['qty'].sum()
print(name_total_quantity)

# Displayed unmatched vendor as "Unknown" instead of leaving it as unmatched or NaN because it reads easier for the viewer
# and still allows them to use the quantity associated with that vendor: 3.

           name  qty
0     Cav Merch    1
1  Hoos Burgers    3
2       Unknown    3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
zone_total_quantity = merged.groupby('zone', dropna=False)['qty'].sum()
print(zone_total_quantity)
print()
print('Sum of zone totals:', sum(zone_total_quantity))
# The unmatched vendor has no zone, so the difference went to the V-07 unmatched vendor because its zone was filled with NaN.

zone
A      4
NaN    3
Name: qty, dtype: int64

Sum of zone totals: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

for order in ragged_json:
    order.setdefault('lines', [])

import pandas as pd

ragged_payload = pd.json_normalize(ragged_json, record_path= 'lines', meta=['order', 'vendor_id'
  ], errors='ignore')

ragged_payload['qty_given'] = ragged_payload['qty'].notna()

print(ragged_payload)
print()
print('Order 5 does not show up as a row because it has no items.')


    item  qty order vendor_id  qty_given
0   Soda  2.0     4      V-01       True
1  Fries  NaN     6      V-01      False

Order 5 does not show up as a row because it has no items.


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(merged) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) The "one row per line item" shape is useful in this lab because there aren't duplicates and each item gets its own row to easily calculate sums of quanitites without being stuck in a list. Whereas if we did one row per order, finding the total quanitity per item would be difficult due to complex list nesting.

b)A missing quantity entails that the value there is unknown, whereas a quantity of zero means you know it is a value of 0. If you conflate them, it might make your data look correct when its not: so in the case of this lab, if we called the qty of fries 0, then we'd be saying that absolutely no fries were sold, when in reality we just don't know how many fries were sold--it could be 0, or any quantity.